# 02 — Exploratory Data Analysis (EDA)
**Dataset:** owid-energy-cleaned.csv (output of notebook 01)
**Purpose:** Identify key patterns, distributions, correlations, and country-level trends in the global energy transition.

---
Sections:
1. Univariate Analysis — distributions of key variables
2. Correlation Analysis — relationships between energy, wealth, and emissions
3. Top/Bottom Countries — who is leading and lagging
4. Regional Trends — continental-level trajectories
5. Growth Analysis — fastest-transforming nations

## Setup & Data Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings, os

warnings.filterwarnings('ignore')
sns.set_palette('Set2')
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'font.family': 'DejaVu Sans'})

DATA_CLEAN = os.path.join('..', 'data', 'owid-energy-cleaned.csv')
df = pd.read_csv(DATA_CLEAN)

# Exclude 'World' aggregate from country-level analyses
df_countries = df[df['country'] != 'World'].copy()
df_world     = df[df['country'] == 'World'].copy()

print(f'Loaded: {df.shape} | Countries: {df_countries["country"].nunique()} | Years: {df["year"].min()}–{df["year"].max()}')

---
## Section 1 — Univariate Analysis (2022 snapshot)
Examine the distribution of three key variables using 2022 data — the most recent complete year:
- `renewables_share_energy`: how green is each country's energy mix?
- `co2_per_capita`: emissions burden per person
- `gdp_per_capita`: economic development level

Histograms with KDE curves reveal skewness and outliers that inform modeling choices later.

In [ ]:
df_2022 = df_countries[df_countries['year'] == 2022].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

variables = [
    ('renewables_share_energy', 'Renewables Share (%)', '#27ae60'),
    ('co2_per_capita',          'CO₂ per Capita (t)',   '#e74c3c'),
    ('gdp_per_capita',          'GDP per Capita (USD)', '#3498db'),
]

for ax, (col, label, color) in zip(axes, variables):
    data = df_2022[col].dropna()
    sns.histplot(data, ax=ax, kde=True, color=color, bins=20, alpha=0.6, edgecolor='white')
    ax.set_title(f'{label}\n(2022, n={len(data)} countries)', fontsize=12, fontweight='bold')
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    # Stats annotation
    stats_text = f'Mean: {data.mean():.1f}\nMedian: {data.median():.1f}\nStd: {data.std():.1f}\nSkew: {stats.skew(data):.2f}'
    ax.text(0.97, 0.97, stats_text, transform=ax.transAxes, fontsize=9,
            va='top', ha='right', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

fig.suptitle('Univariate Distributions — 2022 Country Snapshot', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../assets/eda_univariate.png', bbox_inches='tight')
plt.show()
print('Key insight: Renewables distribution is bimodal — many low-renew (fossil-heavy) and some very high-renew (hydro-dominant) nations.')

---
## Section 2 — Correlation Analysis
A correlation matrix help to detect linear relationships between key dimensions:
- Does wealth drive more renewables, or do richer countries use more fossil fuels?
- Does higher energy intensity correlate with more emissions?
- Are renewables and CO₂ negatively correlated as expected?

Pearson correlation used on 2022 data and annotate the heatmap for clarity.

In [ ]:
corr_cols = ['renewables_share_energy', 'gdp_per_capita', 'co2_per_capita',
             'energy_per_gdp', 'population', 'fossil_share_energy', 'clean_share_energy']

corr_labels = {
    'renewables_share_energy': 'Renewables Share',
    'gdp_per_capita':          'GDP per Capita',
    'co2_per_capita':          'CO₂ per Capita',
    'energy_per_gdp':          'Energy Intensity',
    'population':              'Population',
    'fossil_share_energy':     'Fossil Share',
    'clean_share_energy':      'Clean Share',
}

corr_data = df_2022[corr_cols].rename(columns=corr_labels)
corr_matrix = corr_data.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix — Key Energy Variables (2022)', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../assets/eda_correlation.png', bbox_inches='tight')
plt.show()

### Top 3 Correlation Findings
1. **Renewables Share ↔ Fossil Share (≈ -1.0):** Near-perfect inverse relationship — as expected, they sum to ~100%.
2. **CO₂ per Capita ↔ GDP per Capita (positive):** Wealthier nations tend to emit more, though outliers like Norway show decoupling is achievable.
3. **Energy Intensity ↔ Renewables Share (negative):** Countries with more renewables tend to use energy more efficiently per unit of GDP — suggesting renewable transitions drive efficiency gains.

---
## Section 3 — Top & Bottom Countries (2022)
- Top 10 by renewables share — predominantly hydro-rich or wind/solar leaders
- Bottom 10 — typically Gulf states or coal-dependent developing economies

Excluded micro-nations (population < 1M) to ensure meaningful comparisons.

In [ ]:
df_2022_large = df_2022[df_2022['population'] >= 1_000_000].copy()
df_sorted = df_2022_large.sort_values('renewables_share_energy', ascending=False).dropna(subset=['renewables_share_energy'])

top10    = df_sorted.head(10)
bottom10 = df_sorted.tail(10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Top 10
colors_top = sns.color_palette('Greens_r', 10)
bars = ax1.barh(top10['country'], top10['renewables_share_energy'], color=colors_top, edgecolor='white')
ax1.set_xlabel('Renewables Share (%)')
ax1.set_title('🏆 Top 10 Countries\nby Renewables Share (2022)', fontsize=12, fontweight='bold')
ax1.set_xlim(0, 110)
for bar, val in zip(bars, top10['renewables_share_energy']):
    ax1.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=9)
ax1.invert_yaxis()

# Bottom 10
colors_bot = sns.color_palette('Reds', 10)
bars2 = ax2.barh(bottom10['country'], bottom10['renewables_share_energy'], color=colors_bot, edgecolor='white')
ax2.set_xlabel('Renewables Share (%)')
ax2.set_title('⚠️ Bottom 10 Countries\nby Renewables Share (2022)', fontsize=12, fontweight='bold')
ax2.set_xlim(0, 15)
for bar, val in zip(bars2, bottom10['renewables_share_energy']):
    ax2.text(val + 0.1, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=9)
ax2.invert_yaxis()

plt.tight_layout()
plt.savefig('../assets/eda_top_bottom.png', bbox_inches='tight')
plt.show()

---
## Section 4 — Regional Trends (1990–2022)
Aggregated by continent and year to see macro-level trajectories.
Each continent represents a different combination of resource endowment, policy environment, and economic development.
The multi-line chart lets us compare acceleration rates.

In [ ]:
# Exclude 'World' from continent analysis
regional = df_countries.groupby(['continent', 'year'])['renewables_share_energy'].mean().reset_index()

continents = [c for c in regional['continent'].unique() if c != 'World']
palette = sns.color_palette('Set2', len(continents))

fig, ax = plt.subplots(figsize=(12, 6))
for continent, color in zip(continents, palette):
    data = regional[regional['continent'] == continent]
    ax.plot(data['year'], data['renewables_share_energy'], label=continent,
            linewidth=2.2, color=color, marker='o', markersize=2)

ax.axvline(2015, color='gray', linestyle='--', alpha=0.6, linewidth=1)
ax.text(2015.3, ax.get_ylim()[1]*0.95, 'Paris Agreement', fontsize=9, color='gray')

ax.set_title('Average Renewable Energy Share by Continent (1990–2022)', fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Avg. Renewables Share (%)')
ax.legend(title='Continent', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.grid(True, alpha=0.3)
sns.despine()
plt.tight_layout()
plt.savefig('../assets/eda_regional_trends.png', bbox_inches='tight')
plt.show()
print('Finding: South America and Africa lead due to large hydro capacity; Europe shows steepest growth post-2010.')

---
## Section 5 — Growth Analysis (2000–2022)
Which countries have transformed their energy mix the most in 22 years?
Computed the absolute percentage-point change in renewables share between 2000 and 2022.
Countries with small populations or missing data in either year are excluded.

In [ ]:
pivot = df_countries[df_countries['year'].isin([2000, 2022])].pivot_table(
    index='country', columns='year', values='renewables_share_energy'
).dropna()

pivot['growth'] = pivot[2022] - pivot[2000]
top_growers = pivot.nlargest(15, 'growth').reset_index()

fig, ax = plt.subplots(figsize=(12, 7))
colors = ['#27ae60' if g > 0 else '#e74c3c' for g in top_growers['growth']]
bars = ax.bar(top_growers['country'], top_growers['growth'], color=colors, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, top_growers['growth']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'+{val:.1f}pp', ha='center', va='bottom', fontsize=8.5, fontweight='bold')

ax.set_title('Top 15 Countries: Renewables Share Growth 2000→2022\n(Percentage-Point Change)', fontsize=13, fontweight='bold')
ax.set_ylabel('Growth in Renewables Share (pp)')
ax.set_xlabel('')
plt.xticks(rotation=35, ha='right')
ax.grid(axis='y', alpha=0.3)
sns.despine()
plt.tight_layout()
plt.savefig('../assets/eda_growth.png', bbox_inches='tight')
plt.show()

---
## Section 6 — Energy Source Breakdown (2022)
A stacked bar chart showing the relative composition of energy sources per continent in 2022.

In [ ]:
source_cols = ['coal_share_energy', 'oil_share_energy', 'gas_share_energy',
               'hydro_share_energy', 'wind_share_energy', 'solar_share_energy']
source_labels = ['Coal', 'Oil', 'Gas', 'Hydro', 'Wind', 'Solar']
source_colors = ['#2c3e50', '#7f8c8d', '#95a5a6', '#2980b9', '#27ae60', '#f39c12']

cont_2022 = df_countries[df_countries['year'] == 2022].groupby('continent')[source_cols].mean()
cont_2022.columns = source_labels

ax = cont_2022.plot(kind='bar', stacked=True, figsize=(12, 6),
                    color=source_colors, edgecolor='white', linewidth=0.5)
ax.set_title('Energy Source Mix by Continent (2022)', fontsize=13, fontweight='bold')
ax.set_xlabel('Continent')
ax.set_ylabel('Average Share (%)')
ax.legend(title='Source', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig('../assets/eda_source_breakdown.png', bbox_inches='tight')
plt.show()
print('Finding: Asia relies heavily on coal; South America and Africa are dominated by hydro; Middle East still almost 100% fossil.')